# 🚀 Entrenamiento MoE (DPO) para CPT Simulator v5
Este notebook está diseñado para ejecutarse en la **GPU gratuita de Google Colab (T4)**. Tomará el dataset consolidado de tu simulador local y entrenará un adaptador LoRA para tu modelo unificado.

In [ ]:
!pip install -q -U torch transformers peft trl datasets accelerate bitsandbytes

## 1. Subir el Dataset
Sube el archivo `dpo_dataset_moe_final.jsonl` generado por tu simulador local.

In [ ]:
from google.colab import files
print("Sube tu archivo dpo_dataset_moe_final.jsonl:")
uploaded = files.upload()

## 2. Preparar el Modelo y el Dataset

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import DPOTrainer

# Usaremos el Qwen2.5-0.5B (equivalente al 0.6b local)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Cargar dataset
dataset = load_dataset("json", data_files="dpo_dataset_moe_final.jsonl", split="train")

# Formatear para DPO
def format_dpo(example):
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }
dataset = dataset.map(format_dpo)

# Cargar Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Cargar Modelo con optimización de memoria (bfloat16 si está disponible)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)
model_ref = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)

# Configurar Adaptador LoRA (El 'Cerebro' que vamos a entrenar)
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 3. Iniciar el Entrenamiento DPO (Alineación de Preferencias)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-moe-adapter",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=3,
    logging_steps=10,
    remove_unused_columns=False,
    optim="adamw_torch",
)

trainer = DPOTrainer(
    model=model,
    ref_model=model_ref,
    args=training_args,
    beta=0.1, # Qué tanto nos alejamos del modelo original
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print("🔥 Iniciando entrenamiento DPO...")
trainer.train()

## 4. Guardar y Descargar el Adaptador LoRA

In [ ]:
trainer.model.save_pretrained("cpt_moe_adapter_final")
tokenizer.save_pretrained("cpt_moe_adapter_final")

# Comprimir para descargar fácilmente
!zip -r cpt_moe_adapter_final.zip cpt_moe_adapter_final

print("✅ Entrenamiento finalizado. Descargando adaptador...")
files.download('cpt_moe_adapter_final.zip')
print("Una vez descargado, puedes cargarlo en Ollama/Llama.cpp en tu PC local.")